<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c11-header.png" alt="Nextia Learning · Evaluating and Testing AI Systems" width="100%">

# Calibrate a model judge against human ratings

**[Calibrate a model judge against human ratings](https://learning.nextia-ai.com/courses/evals/m07/calibrate-a-model-judge/)** · Evaluating and Testing AI Systems · Case study · [learning.nextia-ai.com](https://learning.nextia-ai.com)

A model judge can compare two answers in seconds. Before you let it decide anything, you measure it against people.
In this notebook you use 834 real votes by human judges on 320 pairs of chatbot answers (MT-Bench).
You also use the recorded verdicts of two model judges, asked in both orders. You measure how often people agree with each other,
how often each judge agrees with them, and where the judge fails. Then you write a **judge policy**: what the judge may decide
alone, and what must go to a person. At the end, you test the policy once, on questions it has never seen.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page; the notebook runs in under a minute |
| **You need** | No account and no key: the model judgments were recorded once and are replayed here |
| **Before this** | [Calibrate reviewers](https://learning.nextia-ai.com/courses/evals/m04/calibrate-reviewers/) and [Model-based judging](https://learning.nextia-ai.com/courses/evals/m04/model-based-judging/) |
| **Tested** | Python 3.12 with numpy 2.5.3 and matplotlib, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/evals/m07/calibrate-a-model-judge/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C11/M07-L01-calibrate-a-model-judge/calibrate-a-model-judge-solution.ipynb).

## Setup: get the data

Run the next cell. It makes a folder `judge-calibration` and downloads two files from the course's labs repository into `judge-calibration/data/`. One is the fixed sample of MT-Bench items with the human votes (about 1 MB). The other has the recorded judgments of the two model judges (about 1.3 MB). It checks the checksum (SHA-256) of each file. It needs an internet connection and no account.

You should see `Ready:` and the two files. If a copy of a file is next to this notebook, the cell uses that copy.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `judge-calibration` (in Colab: the **Files** panel on the left), then run the cell again.

**Source and licence.** MT-Bench human judgments, LMSYS (Zheng et al., 2023, "Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena", arXiv:2306.05685), [huggingface.co/datasets/lmsys/mt_bench_human_judgments](https://huggingface.co/datasets/lmsys/mt_bench_human_judgments), licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Categories, reference answers and the judge prompts come from [lm-sys/FastChat](https://github.com/lm-sys/FastChat) (Apache-2.0). Changes: the course selected 320 first-turn items with votes from 2 or more human judges (40 per category), kept some fields, wrote every label in a fixed model order, and added the category, a dev/holdout split by question and, for questions 101–130, the reference answer. The model judgments were recorded by the course on Azure. See `dataset.md`.

In [ ]:
# Setup: download the two data files into judge-calibration/data/ and check them. Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C11/M07-L01-calibrate-a-model-judge/data/"
FILES = {
    "mtb_judge_sample.jsonl": "7a6a3e2f52dca0d05a65669def84d5d55637be9d74388e2ba50b22afbc1cbf58",
    "judgments.jsonl": "48c91b7c4bc08d93b6157277c842ee6a83811f8173c1a315322bb5b3a9969f7c"
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
HERE, WORK = Path.cwd(), Path("judge-calibration").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for name, expected in FILES.items():
    path = WORK / "data" / name
    if not (path.exists() and sha256(path) == expected):
        if (HERE / name).exists():  # a copy next to the notebook
            shutil.copy(HERE / name, path)
        else:
            try:
                urllib.request.urlretrieve(URL + name, path)
            except OSError:  # Python without certificates (python.org on macOS): try curl
                if subprocess.run(["curl", "-fsSL", "-o", str(path), URL + name]).returncode:
                    raise SystemExit(f"Could not download {URL + name}. Check your internet connection.")
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder judge-calibration and run again.")
os.chdir(WORK)
print("Ready:", ", ".join(f"data/{n} ({Path('data', n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the measures of the lesson, so the later cells stay short:

- `kappa(pairs)` takes a list of label pairs, for example `[("A", "A"), ("A", "tie")]`, and returns the **percent agreement** and **Cohen's kappa**, as in [Calibrate reviewers](https://learning.nextia-ai.com/courses/evals/m04/calibrate-reviewers/).
- `interval(per_item, which)` gives a 95% **bootstrap interval**, as in [Uncertainty and slices](https://learning.nextia-ai.com/courses/evals/m05/uncertainty-and-slices/). It resamples the *items* 2,000 times with seed 2026, because the votes on one item are not independent.
- `expect_hash()` checks your answers without showing them.

Run the cell. It prints nothing.

In [ ]:
# Helpers: agreement, kappa, bootstrap intervals, and checks that hide the answer.
import hashlib, itertools, json
from collections import Counter
import numpy as np

LABELS = ["A", "B", "tie"]

def kappa(pairs):
    """(percent agreement, Cohen's kappa) of a list of (label, label) pairs."""
    n = len(pairs)
    if n == 0:  # a resample can draw only items without a pair
        return float("nan"), float("nan")
    observed = sum(a == b for a, b in pairs) / n
    first, second = Counter(a for a, _ in pairs), Counter(b for _, b in pairs)
    chance = sum(first[k] * second[k] for k in LABELS) / n**2
    return observed, (observed - chance) / (1 - chance) if chance < 1 else float("nan")

def interval(per_item, which=0, seed=2026, resamples=2000):
    """per_item: one list of pairs per item. -> (estimate, low, high) for agreement (0) or kappa (1)."""
    estimate = kappa([p for pairs in per_item for p in pairs])[which]
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(resamples):
        picked = rng.choice(len(per_item), len(per_item))
        values.append(kappa([p for i in picked for p in per_item[i]])[which])
    low, high = np.nanpercentile(values, [2.5, 97.5])
    return round(estimate, 3), round(float(low), 3), round(float(high), 3)

def fingerprint(value):
    def norm(v):
        if isinstance(v, (float, np.floating)):
            return round(float(v), 3)
        if isinstance(v, (np.integer,)):
            return int(v)
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items())}
        return v
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect_hash(what, yours, expected):
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data: one question, two answers, several people

Each line of `mtb_judge_sample.jsonl` is one **item**. It has a question from MT-Bench, the first answers of two chatbots of 2023, and the votes of the people who compared them. A vote is `"A"` (the answer of `model_1` is better), `"B"` (the answer of `model_2` is better) or `"tie"`. `model_1` is always the model whose name comes first in alphabetical order. So a label means the same thing on every line.

Who voted? The paper says: "58 expert-level human labelers … mostly graduate students", paid to judge 20 questions each, plus some of the paper's authors. In the data they have pseudonyms such as `expert_17` and `author_4`. These are real human ratings; the model judges come later, and the notebook keeps them apart.

The items are split **by question**. In each category, 5 questions are `dev`: there you measure the judges and write the policy. The other 5 are `holdout`: you test the policy on them once, at the end.

Run the next cell.

In [ ]:
items = [json.loads(line) for line in open("data/mtb_judge_sample.jsonl", encoding="utf-8")]
dev = [it for it in items if it["split"] == "dev"]
holdout = [it for it in items if it["split"] == "holdout"]
votes = [r for it in items for r in it["human_ratings"]]
print(len(items), "items |", len(dev), "dev |", len(holdout), "holdout")
print(len(votes), "human votes from", len({r["rater_id"] for r in votes}), "people")
print("votes per item:", dict(sorted(Counter(len(it["human_ratings"]) for it in items).items())))
print("categories:", dict(Counter(it["category"] for it in items)))

> **Check.** You should see `320 items | 155 dev | 165 holdout` and `834 human votes from 65 people`. Most items have 2 votes (187 of 320); a few have up to 7. Every category has 40 items.

Look at one item. The next cell prints the question, the start of each answer, and the votes.

In [ ]:
item = dev[0]
print(item["item_id"], "|", item["category"], "|", item["split"])
print("Question:", item["question"][:300])
print("\nAnswer 1 (" + item["model_1"] + "):", item["answer_1"][:300], "...")
print("\nAnswer 2 (" + item["model_2"] + "):", item["answer_2"][:300], "...")
print("\nVotes:", [(r["rater_id"], r["label"]) for r in item["human_ratings"]])

### The human majority

To compare a judge with "the people", you need one label per item. The **majority** is the label with the most votes. Sometimes two labels have the same, highest number of votes, for example one `A` and one `B`. Then the item has **no majority**, and it is left out of the comparison with the majority.

In [ ]:
def majority(item):
    """The label with the most votes, or None when the top labels have the same count."""
    counts = Counter(r["label"] for r in item["human_ratings"]).most_common()
    if len(counts) > 1 and counts[0][1] == counts[1][1]:
        return None
    return counts[0][0]

print("dev majority:", dict(Counter(majority(it) or "no majority" for it in dev)))
print("dev items where every person voted the same:", sum(len({r["label"] for r in it["human_ratings"]}) == 1 for it in dev))

> **Check.** On dev: `A` 46, `B` 49, `no majority` 33, `tie` 27. Every person voted the same on 92 of the 155 dev items.

So on 33 dev items, the people split evenly. Those items are not "hard for the judge": they are hard, full stop.

## 2. The ceiling: how often do two people agree?

A judge cannot be measured against a perfect answer, because there is none. The fair question is: does the judge agree with a person as often as **another person** does? So first take every pair of two people who voted on the same item, and measure their agreement.

In [ ]:
def human_pairs(item):
    """Every pair of two different people on this item."""
    ratings = sorted(item["human_ratings"], key=lambda r: r["rater_id"])
    return [(a["label"], b["label"]) for a, b in itertools.combinations(ratings, 2)]

per_item = [human_pairs(it) for it in dev]
print("pairs of people:", sum(len(p) for p in per_item))
print("agreement:", interval(per_item, 0))
print("kappa:    ", interval(per_item, 1))
print("agreement without ties:", interval([[p for p in ps if "tie" not in p] for ps in per_item], 0))

> **Check.** On dev, 376 pairs of people agree on 0.638 [0.563, 0.711] of them, kappa 0.451 [0.337, 0.554]. Without ties, they agree on 0.834 [0.759, 0.898].

Two people agree only about 64% of the time, and most of the rest is one "tie" against one winner. That is the **ceiling**: a judge that agrees with people this often is as good as a second person. The interval in brackets is wide because there are only 155 dev items.

### Your turn 1

How many **pairs of people** on the dev items gave exactly opposite votes (one `A`, the other `B`)? Write the number in `opposite`.

In [ ]:
# Your turn 1: count the pairs with one "A" and one "B".
opposite = ...
print(opposite)

In [ ]:
expect_hash("opposite", opposite, 'aea92132c4cbeb26')

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 3. The judges, asked twice

The course asked two model judges on Azure, **chat-small** (gpt-6-luna) and **chat-strong** (gpt-6.1-sol), with the MT-Bench paper's own pairwise prompt. The prompt ends: "output your final verdict … "[[A]]" if assistant A is better, "[[B]]" if assistant B is better, and "[[C]]" for a tie." Each pair was asked **twice**: once with answer 1 shown as "Assistant A", once with answer 2 shown as "Assistant A". For reasoning, math and coding, the course also asked the paper's second prompt, which adds a **reference answer** (written by GPT-4 for MT-Bench).

The file `judgments.jsonl` replays every recorded call: the letter that the judge wrote, in the places it saw them, and its full text. Models change fast; these are a snapshot of 9 October 2026. The method lasts.

The dataset also has the verdicts of the paper's own judge, **GPT-4 in 2023**, asked in both orders as well (`gpt4_2023`). It is a third judge for comparison.

First, turn each letter into the fixed direction. In order `"12"`, "A" means answer 1. In order `"21"`, the places are swapped, so "A" means answer 2.

In [ ]:
judgments = [json.loads(line) for line in open("data/judgments.jsonl", encoding="utf-8")]
print(len(judgments), "recorded judgments |", dict(Counter((j["judge"], j["prompt"]) for j in judgments)))
print("outcomes:", dict(Counter(j["outcome"] for j in judgments)))

def fixed(j):
    """The verdict in the fixed direction: 'A' = answer 1 better, 'B', 'tie', or the outcome if there is no letter."""
    if j["outcome"] != "ok":
        return j["outcome"]
    letter = {"A": "A", "B": "B", "C": "tie"}[j["letter"]]
    if j["order"] == "21" and letter != "tie":
        letter = "B" if letter == "A" else "A"
    return letter

both = {}  # (judge, prompt) -> item_id -> {"12": verdict, "21": verdict}
for j in judgments:
    both.setdefault((j["judge"], j["prompt"]), {}).setdefault(j["item_id"], {})[j["order"]] = fixed(j)

> **Check.** You should see the number of recorded judgments per judge and prompt, and their outcomes: 1,760 judgments, almost all `ok`. Two chat-small judgments are `unparsed`: the judge explained its choice but did not write the `[[A]]` format. There are no `refused` judgments: Azure's content filter did not stop any of these requests.

Now combine the two orders into one verdict. If both orders give the same verdict, that is the verdict. If they differ, the order changed it: the verdict is **inconsistent**, as in [Model-based judging](https://learning.nextia-ai.com/courses/evals/m04/model-based-judging/). For agreement, an inconsistent verdict counts as a tie, as in the paper.

> **Predict.** Out of the 155 dev items, on how many will the order change chat-strong's verdict? And chat-small's? Write your guesses down, then run the cell.

In [ ]:
def combine(v12, v21):
    if "refused" in (v12, v21):
        return "refused"
    if not {v12, v21} <= set(LABELS):
        return "unparsed"
    return v12 if v12 == v21 else "inconsistent"

verdicts = {key: {i: combine(v["12"], v["21"]) for i, v in d.items() if "12" in v and "21" in v} for key, d in both.items()}
verdicts[("gpt-4 (2023)", "pair-v2")] = {it["item_id"]: it["gpt4_2023"]["label"] for it in items}
verdicts[("longer answer", "pair-v2")] = {it["item_id"]: "A" if len(it["answer_1"]) > len(it["answer_2"]) else "B" for it in items}
for judge in ["chat-small", "chat-strong", "gpt-4 (2023)"]:
    print(f"{judge:13}", dict(Counter(verdicts[(judge, "pair-v2")].get(it["item_id"], "missing") for it in dev)))

> **Check.** On the dev items: chat-small `A` 65, `B` 67, `tie` 11, `inconsistent` 12; chat-strong `A` 62, `B` 68, `tie` 17, `inconsistent` 8; gpt-4 (2023) `A` 49, `B` 59, `tie` 20, `inconsistent` 27.

The order changed chat-strong's verdict on 8 of 155 dev items (5%), and chat-small's on 12 (8%). GPT-4 in 2023 changed on 27 (17%): today's judges are much steadier, but not free of order bias.

### Your turn 2

For **chat-small** on the dev items (prompt `pair-v2`), what share of the items with a verdict were inconsistent? Round to 3 decimals and write it in `small_flip_share`.

In [ ]:
# Your turn 2: inconsistent items / items with a verdict, chat-small, pair-v2, dev.
small_flip_share = ...
print(small_flip_share)

In [ ]:
expect_hash("small_flip_share", small_flip_share, '44652f05931c0b02')

## 4. Agreement with people

Two comparisons, each with an interval:

- **With the human majority**, on the dev items that have one. This answers: "Would the judge give the team's usual answer?"
- **With each person**: every human vote against the judge's verdict on the same item. Put it next to the ceiling of section 2 (person against person, the same kind of pairs). This answers: "Is the judge as good as adding one more person?"

In [ ]:
def as_tie(v):
    return "tie" if v == "inconsistent" else v

def vs_majority(verdict, some_items):
    return [[(majority(it), as_tie(verdict[it["item_id"]]))] if majority(it) and verdict.get(it["item_id"]) in LABELS + ["inconsistent"] else []
            for it in some_items]

def vs_people(verdict, some_items):
    return [[(r["label"], as_tie(verdict[it["item_id"]])) for r in it["human_ratings"]]
            if verdict.get(it["item_id"]) in LABELS + ["inconsistent"] else [] for it in some_items]

print(f"{'':15}{'with the majority: agreement':32}{'kappa':24}{'with each person: agreement':30}kappa")
print(f"{'person (ceiling)':15}{'':32}{'':24}{str(interval([human_pairs(it) for it in dev], 0)):30}{interval([human_pairs(it) for it in dev], 1)}")
for judge in ["longer answer", "chat-small", "chat-strong", "gpt-4 (2023)"]:
    v = verdicts[(judge, "pair-v2")]
    print(f"{judge:15}{str(interval(vs_majority(v, dev), 0)):32}{str(interval(vs_majority(v, dev), 1)):24}"
          f"{str(interval(vs_people(v, dev), 0)):30}{interval(vs_people(v, dev), 1)}")

> **Check.** Person with person: 0.638 [0.563, 0.711], kappa 0.451 [0.337, 0.554]. chat-strong with each person: 0.646 [0.579, 0.704]; with the majority 0.738 [0.653, 0.811]. chat-small: 0.639 [0.572, 0.699] and 0.730 [0.642, 0.803]. The longer-answer rule: 0.537 [0.473, 0.605] and 0.582 [0.500, 0.672].

Read it like with like. Against one person, both Azure judges agree as often as another person does: they sit at the ceiling, and so does GPT-4 of 2023. Against the majority, every judge scores higher, because a majority is easier to agree with; never compare that column with the ceiling. The longer-answer rule is clearly lower: the judges read more than length. chat-small and chat-strong are within a point of each other, and chat-small costs less than a tenth as much.

## 5. Does the judge prefer the longer answer?

The paper's prompt says: "Do not allow the length of the responses to influence your evaluation." Check it. For every **decisive** verdict (A or B), is the chosen answer the longer one? Do the same for the human majority, so you know the baseline. People may prefer longer answers too, and sometimes the longer answer is better.

In [ ]:
def longer(it):
    return "A" if len(it["answer_1"]) > len(it["answer_2"]) else "B"

human_majority = {it["item_id"]: majority(it) for it in dev}
for name, v in [("people (majority)", human_majority)] + [(j, verdicts[(j, "pair-v2")]) for j in ["chat-small", "chat-strong", "gpt-4 (2023)"]]:
    decisive = [it for it in dev if v.get(it["item_id"]) in ("A", "B")]
    share = sum(v[it["item_id"]] == longer(it) for it in decisive) / len(decisive)
    print(f"{name:18} picks the longer answer in {share:.0%} of {len(decisive)} decisive verdicts")

> **Check.** People (the majority) pick the longer answer in 75% of their decisive verdicts; chat-strong in 64%, chat-small in 63%, GPT-4 (2023) in 76%.

Longer answers win often, with people too: they are often more complete. Today's judges pick the longer answer less often than the people do. So there is no sign of a length bias beyond the people's own. This is a result for these questions and these judges; check it again for yours.

## 6. Where does the judge fail? By category

An overall number blends the categories. Split it, as in [Uncertainty and slices](https://learning.nextia-ai.com/courses/evals/m05/uncertainty-and-slices/). Keep the counts and the intervals next to each number, because each category has only about 20 dev items.

In [ ]:
CATEGORIES = ["writing", "roleplay", "reasoning", "math", "coding", "extraction", "stem", "humanities"]
print(f"{'category':11}{'items':>6}  {'person vs person':22}{'chat-strong vs majority':26}{'chat-small vs majority':26}flips (strong)")
for cat in CATEGORIES:
    sub = [it for it in dev if it["category"] == cat]
    strong, small = verdicts[("chat-strong", "pair-v2")], verdicts[("chat-small", "pair-v2")]
    flips = sum(strong.get(it["item_id"]) == "inconsistent" for it in sub)
    print(f"{cat:11}{len(sub):>6}  {str(interval([human_pairs(it) for it in sub])):22}"
          f"{str(interval(vs_majority(strong, sub))):26}{str(interval(vs_majority(small, sub))):26}{flips}")

> **Check.** The table has one row per category. It shows the number of dev items, person with person, and each judge with the majority, with intervals. The last column counts the chat-strong verdicts that changed with the order.

The intervals are wide: about 20 items per category. Coding stands out: people agree with each other there more than in any other category, and the judges do not reach them. The [lesson page](https://learning.nextia-ai.com/courses/evals/m07/calibrate-a-model-judge/) has the chart of the same numbers against one person.

### Your turn 3

What is **chat-strong**'s agreement with the human majority on the dev items of the category **extraction**? Give the estimate only, not the interval, with 3 decimals. Write it in `category_agreement`.

In [ ]:
# Your turn 3: the estimate of chat-strong's agreement with the majority, one category, dev.
category_agreement = ...
print(category_agreement)

In [ ]:
expect_hash("category_agreement", category_agreement, '1e181f0934d44144')

## 7. With and without a reference answer

For reasoning, math and coding, a judge that reads two answers may make the same mistake as the answers. The paper's fix is a **reference answer** in the prompt. Compare the same judge, on the same dev items, with and without it. The interval of the **difference** comes from a paired bootstrap: the same resampled items for both.

In [ ]:
ref_dev = [it for it in dev if it["reference"]]

def item_agreement(verdict, some_items):
    return np.array([(majority(it) == as_tie(verdict[it["item_id"]])) if majority(it) and verdict.get(it["item_id"]) in LABELS + ["inconsistent"]
                     else np.nan for it in some_items], dtype=float)

for judge in ["chat-small", "chat-strong"]:
    without = item_agreement(verdicts[(judge, "pair-v2")], ref_dev)
    with_ref = item_agreement(verdicts[(judge, "pair-math-v1")], ref_dev)
    ok = ~np.isnan(without) & ~np.isnan(with_ref)
    a, b = without[ok], with_ref[ok]
    rng = np.random.default_rng(2026)
    diffs = [np.mean(b[s] - a[s]) for s in (rng.integers(0, len(a), len(a)) for _ in range(2000))]
    low, high = np.percentile(diffs, [2.5, 97.5])
    print(f"{judge:12} {ok.sum()} items | without {a.mean():.3f} | with the reference {b.mean():.3f} | "
          f"difference {b.mean() - a.mean():+.3f} [{low:+.3f}, {high:+.3f}]")

> **Check.** chat-small: difference +0.000 [-0.082, +0.082]; chat-strong: -0.020 [-0.060, +0.000].

The reference answer does not help here: the differences are 0 or slightly negative, and the intervals include 0. The 2023 answers' mistakes are easy for today's judges to see without help. A reference still costs a person's work to write and check, so here it is not worth it.

## 8. A failure case: asking only once

> **Warning.** Most judge set-ups ask once, with the answers in a fixed order. Run the next cell: it measures chat-strong on dev with **only** the first order, `"12"`, as if the second call had never been made.

In [ ]:
once = {i: v["12"] for i, v in both[("chat-strong", "pair-v2")].items() if v["12"] in LABELS}
print("asked once:  ", dict(Counter(once.get(it["item_id"]) for it in dev)))
print("agreement with the majority, asked once: ", interval(vs_majority(once, dev)))
print("agreement with the majority, both orders:", interval(vs_majority(verdicts[("chat-strong", "pair-v2")], dev)))

> **Check.** Asked once, chat-strong gives `A` 63, `B` 75, `tie` 17: no inconsistent verdicts at all. Agreement with the majority: 0.730 [0.645, 0.803] asked once, 0.738 [0.653, 0.811] with both orders.

The average hardly moves. What changes is what you know: asked once, the 8 items where the place decided look exactly as certain as the others. Those items are the ones a person should read.

## 9. The judge policy

Now write down what the judge may decide alone, from dev only:

1. Use the judge that agrees with the human majority more often on dev: **chat-strong**.
2. For reasoning, math and coding, use the reference prompt if, on dev, it agrees with the majority at least as often.
3. A category goes to a person when the judge agrees with **one person** more than **0.05** less often than one person agrees with another, in that category. Like with like: the judge's pairs and the people's pairs are the same kind.
4. Every item whose verdict changed with the order goes to a person. So does every refused or unreadable judgment.

The first version of rule 3 compared the judge's agreement with the *majority* with the agreement of two *single* people. A majority is easier to agree with, so that rule favoured the judge. It was fixed on dev, before any holdout number was computed; the [lesson page](https://learning.nextia-ai.com/courses/evals/m07/calibrate-a-model-judge/) tells the story.

The next cell computes the policy.

In [ ]:
def write_policy(judge, margin=0.05):
    prompt, alone = {}, {}
    for cat in CATEGORIES:
        sub = [it for it in dev if it["category"] == cat]
        prompt[cat] = "pair-v2"
        if cat in ("reasoning", "math", "coding"):
            if interval(vs_majority(verdicts[(judge, "pair-math-v1")], sub))[0] >= interval(vs_majority(verdicts[(judge, "pair-v2")], sub))[0]:
                prompt[cat] = "pair-math-v1"
        judge_agreement = interval(vs_people(verdicts[(judge, prompt[cat])], sub))[0]
        people_agreement = interval([human_pairs(it) for it in sub])[0]
        alone[cat] = judge_agreement >= round(people_agreement - margin, 3)
        print(f"{cat:11} {prompt[cat]:13} judge vs person {judge_agreement:.3f} | person vs person {people_agreement:.3f} -> "
              f"{'judge alone' if alone[cat] else 'a person'}")
    return {"judge": judge, "prompt": prompt, "alone": alone}

policy = write_policy("chat-strong")

def route(item, policy):
    v = verdicts[(policy["judge"], policy["prompt"][item["category"]])].get(item["item_id"])
    if v is None or v in ("refused", "unparsed"):
        return "person: no verdict", v
    if v == "inconsistent":
        return "person: the order changed the verdict", v
    if not policy["alone"][item["category"]]:
        return "person: category", v
    return "judge", v

> **Check.** chat-strong is the judge; reasoning and math use the reference prompt (it was as good on dev, not better); coding goes to a person; every other category is decided by the judge alone, except the items whose verdict changed with the order.

## 10. The test, once: the holdout questions

The holdout items are 40 other questions. Nothing in the policy was chosen on them. Apply the policy once, and measure two things: how much work the judge takes over, and how well it does there.

In [ ]:
decisions = {it["item_id"]: route(it, policy) for it in holdout}
print("decisions:", dict(Counter(d for d, _ in decisions.values())))
alone = [it for it in holdout if decisions[it["item_id"]][0] == "judge"]
decided = {i: v for i, (d, v) in decisions.items()}
print(f"the judge decides alone on {len(alone)} of {len(holdout)} items ({len(alone) / len(holdout):.0%})")
print("there, agreement with the majority:", interval(vs_majority(decided, alone)), "| kappa", interval(vs_majority(decided, alone), 1))
print("there, person vs person:           ", interval([human_pairs(it) for it in alone]))
print("judge on every holdout item, no policy:", interval(vs_majority(verdicts[(policy["judge"], "pair-v2")], holdout)))

> **Check.** Decisions: judge 139, person: category 17, person: the order changed the verdict 9. The judge decides 139 of 165 items alone (84%). There it agrees with the majority 0.766 [0.688, 0.845], kappa 0.620 [0.499, 0.741]; person with person on the same items: 0.657 [0.568, 0.745].

On new questions, the judge takes over about 84% of the work, and where it decides, it agrees with one person 0.673 [0.610, 0.736]: as often as people agree with each other. The policy held. It is one test on 165 items, so read the intervals, not only the points.

### One new item, end to end

A new comparison arrives. Here it is one holdout item; in real use it is a pair of answers from this week's model versions. The policy decides who judges it.

In [ ]:
new = next(it for it in holdout if it["item_id"] == "mtb-q116-gpt-4-vs-llama-13b")
decision, verdict = route(new, policy)
print(new["category"], "|", new["question"][:160])
print("judge in order 12:", both[(policy["judge"], policy["prompt"][new["category"]])][new["item_id"]]["12"],
      "| in order 21:", both[(policy["judge"], policy["prompt"][new["category"]])][new["item_id"]]["21"])
print("decision:", decision, "| verdict:", verdict, "| the people said:", [r["label"] for r in new["human_ratings"]])

> **Check.** A math question, `x+y = 4z, x*y = 4z^2, express x-y in z`. The judge gave the same verdict in both orders (`A`, answer 1, gpt-4), math is a category the judge may decide, so the decision is `judge`. All 4 people who voted agree with it.

### Your turn 4

Change one thing: make the policy **stricter**, with a margin of 0 instead of 0.05, on the same dev items. Then the judge must agree with people at least as often as people agree with each other. How many **holdout** items would the judge then decide alone? Write the number in `alone_strict`. Then explain in one sentence what you gain and what you pay.

In [ ]:
# Your turn 4: a stricter policy (margin 0), then count the holdout items the judge decides alone.
alone_strict = ...
print(alone_strict)

In [ ]:
expect_hash("alone_strict", alone_strict, 'd6d824abba4afde8')

## Recap

- People disagree: on these items, two human judges agree on about 64% of the votes (kappa 0.451). That is the ceiling for any judge.
- Against one person, chat-strong (65%) and chat-small (64%) agree as often as another person does (64%), except in coding.
- Ask a pairwise judge in **both orders**. One order hides the cases where the place, not the content, decided.
- A **judge policy** says where the judge may decide alone, from numbers on dev, and it is tested once on holdout.

The [lesson page](https://learning.nextia-ai.com/courses/evals/m07/calibrate-a-model-judge/) has the full story, the diagrams, the knowledge checks and how to make this a portfolio piece.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Evaluate summaries against expert ratings](https://learning.nextia-ai.com/courses/evals/m07/evaluate-summaries-against-expert-ratings/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/evals/m07/calibrate-a-model-judge/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The human votes are real (MT-Bench, CC BY 4.0); the model judgments were recorded by the course on Azure.